In [ ]:
!pip install Sastrawi
!pip install nltk
!pip install matplotlib

In [ ]:
import pandas as pd
import nltk
nltk.download('stopwords')
nltk.download("punkt_tab")

In [ ]:
dir = "D:\***/topic-modelling-LDA-skripsi-matematika"

# Visualisasi

In [ ]:
UINSunanAmpel = pd.read_csv(f"{dir}/dataset/mtk_uinsa2.csv", on_bad_lines='skip')

In [ ]:
llDF = pd.concat([UINSunanAmpel, UINSunanGunungDjati, UINSyahidHida, UINSiberSandi, UINSunanKalijaga, UINMaliki, UINWalisongo])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
allDF['year'].value_counts().sort_index(ascending=True).plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Dokumen Berdasarkan Tahun', fontsize=16)
plt.xlabel('Jumlah', fontsize=12)
plt.ylabel('Tahun', fontsize=12)
plt.show()

# Translate

In [ ]:
!pip install deep-translator
!pip install langdetect

In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
from langdetect import detect

# Fungsi untuk deteksi & translate bahasa
def translate_to_indonesian(text):
    try:
        if detect(text) != 'id':  # Cek apakah bukan Bahasa Indonesia
            return GoogleTranslator(source='auto', target='id').translate(text)
        else:
            return text  # Kalau sudah Bahasa Indonesia, biarkan
    except:
        return text  # Kalau error, biarkan teks asli

In [ ]:
# Terapkan ke field "abstrak"
UINSunanAmpel["abstract"] = UINSunanAmpel["abstract"].astype(str).apply(translate_to_indonesian)

UINSunanAmpel2 = UINSunanAmpel.copy()

In [ ]:
def translate_arab_to_indonesian(text):
    try:
        lang = detect(text)  # Deteksi bahasa
        if lang == 'ar':  # Jika bahasa Arab
            return GoogleTranslator(source='ar', target='id').translate(text)
        elif lang != 'id':  # Jika bukan Bahasa Indonesia atau Arab
            return GoogleTranslator(source='auto', target='id').translate(text)
        else:
            return text  # Jika sudah Bahasa Indonesia, biarkan
    except:
        return text  # Kalau error, biarkan teks asli

# Pre-Processing

In [ ]:
import re, string, unicodedata  #modul regular expression
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk
from nltk import word_tokenize, sent_tokenize  #Paket ini membagi teks input menjadi kata-kata.,
from nltk.corpus import stopwords

# create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# stemmed
def stemmed_wrapper(term):
    return stemmer.stem(term)

def removeStopword(stringText):
    stop_list = ["yang", "dan", "atau", "adalah", "dari", "untuk", "pada", "dengan", "bahwa", "ini", "itu", "tersebut",
    "matematika", "metode", "teori", "konsep", "rumus", "persamaan", "fungsi", "himpunan", "bilangan",
    "variabel", "konstanta", "koefisien", "teorema", "lemma", "definisi", "proposisi", "bukti", "contoh",
    "soal", "hasil", "kesimpulan", "membahas", "menjelaskan", "menunjukkan", "menerapkan", "menggunakan",
    "diketahui", "diperoleh", "kemudian", "selanjutnya", "berikut", "sehingga", "berdasarkan", "terhadap",
    "integral", "limit", "turunan", "matriks", "vektor", "sistem", "linear", "nonlinier", "aljabar",
    "geometri", "probabilitas", "statistik"]
    stop_words = set(stopwords.words('indonesian')).union(stop_list)
    word_tokens = word_tokenize(stringText)
    filtered_sentence = [w for w in word_tokens if not w in stop_words]
    filtered_sentence = [w for w in filtered_sentence if len(w) > 3]
    return filtered_sentence

#remove sentence which contains only one word
def removeSentence(stringText):
    word = stringText.split()
    wordCount = len(word)
    if(wordCount<=1):
        stringText = ''
    return stringText

def cleaning(stringText):
    #remove non-ascii
    stringText = unicodedata.normalize('NFKD', stringText).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    #remove URLs
    stringText = re.sub(r'(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:\'".,<>?«»“”‘’]))', '', stringText)
    #remove punctuations
    stringText = re.sub(r'[^\w]|_',' ',stringText)
    #remove digit from stringTexting
    stringText = re.sub(r"\S*\d\S*", "", stringText).strip()
    #remove digit or numbers
    stringText = re.sub(r"\b\d+\b", " ", stringText)
    #to lowercase
    stringText = stringText.lower()
    #Remove additional white spaces
    stringText = re.sub(r'[\s]+', ' ', stringText)
    return stringText

def preprocessing(stringText):
    stringText = removeSentence(stringText)
    stringText = cleaning(stringText)
    stringText = removeStopword(stringText)
    stringText = [stemmed_wrapper(word) for word in stringText]
    return " ".join(stringText)

In [ ]:
len(UINSunanAmpel2)

UINSunanAmpel2["abstract"] = UINSunanAmpel2["abstract"].apply(preprocessing)

In [ ]:
UINSunanAmpel2.to_csv(f'{dir}/preprocessing/preprocessing_UINSunanAmpel.csv', index=False)

# Bigram and Trigram

In [ ]:
!pip install pyLDAvis

In [ ]:
dir = "D:\***/topic-modelling-LDA-skripsi-matematika"

import pandas as pd
UINAmpel = pd.read_csv(f"{dir}/preprocessing/preprocessing_UINSunanAmpel.csv")

In [ ]:
allDF = pd.concat([UINMaliki, UINSyarif, UINKalijaga, UINDjati, UINAmpel, UINSiber, UINWalisongo])

In [ ]:
text = allDF["abstract"]
text_list = [str(i).split() for i in text if pd.notna(i)]
print(len(text_list))
print(text_list)

In [ ]:
import gensim
#Create Bigram & Trigram Models
from gensim.models import Phrases
# Add bigrams and trigrams to docs, minimum count 10 means only that appear 10 times or more.
bigram = Phrases(text_list, min_count=10)
trigram = Phrases(bigram[text_list])
for idx in range(len(text_list)):
    for token in bigram[text_list[idx]]:
        if '_' in token:
            # Token is a bigram, add to document.
            text_list[idx].append(token)
    for token in trigram[text_list[idx]]:
        if '_' in token:
            # Token is a bigram, add to document.
            text_list[idx].append(token)

print(text_list)
display(text_list)

In [ ]:
from gensim import corpora, models
# Create a dictionary representation of the documents.
dictionary = corpora.Dictionary(text_list)
dictionary.filter_extremes(no_below=5, no_above=0.2)
#no_below (int, optional) – Keep tokens which are contained in at least no_below documents.
#no_above (float, optional) – Keep tokens which are contained in no more than no_above documents (fraction of total corpus size, not an absolute number).
print(dictionary)

In [ ]:
#https://radimrehurek.com/gensim/tut1.html
#build corpus
# Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
doc_term_matrix = [dictionary.doc2bow(doc) for doc in text_list]
#The function doc2bow converts document (a list of words) into the bag-of-words format

# '''The function doc2bow() simply counts the number of occurrences of each distinct word,
# converts the word to its integer word id and returns the result as a sparse vector.
# The sparse vector [(0, 1), (1, 1)] therefore reads: in the document “Human computer interaction”,
# the words computer (id 0) and human (id 1) appear once;
# the other ten dictionary words appear (implicitly) zero times.'''

print(len(doc_term_matrix))
print(doc_term_matrix[100])
tfidf = models.TfidfModel(doc_term_matrix) #build TF-IDF model
corpus_tfidf = tfidf[doc_term_matrix]

# Latent Dirichlet Allocation (LDA)

In [ ]:
from gensim.models.ldamodel import LdaModel
import pyLDAvis.gensim
import pickle
import pyLDAvis
import os

In [ ]:
pyLDAvis.enable_notebook()

# Pastikan direktori utama ada
base_dir = os.path.join(dir, "hasil", "Skenario 5")
lda_model_dir = os.path.join(base_dir, "LDA model")
ldavis_dir = os.path.join(base_dir, "LDAvis")

os.makedirs(lda_model_dir, exist_ok=True)
os.makedirs(ldavis_dir, exist_ok=True)

for numTopic in range(6, 7):
    for passes in [100, 500, 1000, 5000]:
        print(f"numTopic: {numTopic}, passes: {passes} => Starting")

        # Buat model LDA
        model = LdaModel(corpus=corpus_tfidf, id2word=dictionary, num_topics=numTopic, passes=passes)

        # Simpan model
        model_path = os.path.join(lda_model_dir, f"{dir}/hasil/Skenario 5/LDA model/lda_model_allUIN_{numTopic}_{passes}")
        model.save(model_path)

        # Simpan LDAvis
        LDAvis_data_filepath = os.path.join(ldavis_dir, f"{dir}/hasil/Skenario 5/ldavis_prepared_allUIN_{numTopic}_{passes}.pkl")

        data = pyLDAvis.gensim.prepare(model, corpus_tfidf, dictionary)
        with open(LDAvis_data_filepath, 'wb') as f:
            pickle.dump(data, f)

        # Load kembali data LDAvis (opsional, untuk verifikasi)
        with open(LDAvis_data_filepath, 'rb') as f:
            data = pickle.load(f)

        # Simpan sebagai HTML
        html_path = os.path.join(ldavis_dir, f"{dir}/hasil/Skenario 5/ldavis_allUIN_{numTopic}_{passes}.html")
        pyLDAvis.save_html(data, html_path)

        print(f"numTopic: {numTopic}, passes: {passes} => Success")


# Coherence Value

In [ ]:
import os
from gensim import models
from gensim.models.coherencemodel import CoherenceModel

In [ ]:
base_dir = "D:/Anisa Husna Nuha_09010622004/topic-modelling-LDA-skripsi-matematika" 
skenario_list = [5]  

coherenanceVal = {}

for skenario in skenario_list:
    coherenanceVal[skenario] = []
    for numTopic in range(6, 7):  # bisa kamu ubah kalau mau banyak topik
        listCoherenance = []
        for passes in [100, 500, 1000, 5000]:
            file_path = f"{base_dir}/hasil LDA/LDA all PTKIN/Skenario {skenario}/LDA model/lda_model_allUIN_{numTopic}_{passes}"

            if not os.path.exists(file_path):
                print(f"⚠️ File tidak ditemukan: {file_path}")
                continue  

            try:
                lda = models.LdaModel.load(file_path)
                coherencemodel = CoherenceModel(
                    model=lda, texts=text_list, dictionary=dictionary, coherence='c_v'
                ).get_coherence()
                listCoherenance.append((passes, coherencemodel))
                print(f"Skenario {skenario} | {numTopic} topik | passes={passes} | coherence={coherencemodel:.4f}")
            except Exception as e:
                print(f"❌ Error loading model {file_path}: {e}")

        coherenanceVal[skenario].append({numTopic: listCoherenance})

print("\n📊 Semua nilai coherence berhasil dikumpulkan:")
for skenario, vals in coherenanceVal.items():
    print(f"Skenario {skenario}: {vals}")